# Diagnóstico Estadístico — Validación del Análisis Vintage

Evaluación de la rigurosidad estadística de las proyecciones Chain Ladder y estimaciones de MOB 1.

**Tests implementados:**
1. Estabilidad de factores CL (CV por transición + mínimo de cohortes)
2. Test ADF de estacionariedad sobre la serie MOB 1
3. Selección automática de orden ARIMA con AIC/BIC (auto_arima)
4. Análisis de residuos del modelo seleccionado
5. Propagación de error en la cadena CL

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from IPython.display import display, HTML
from statsmodels.tsa.stattools import adfuller, acf
from statsmodels.stats.diagnostic import acorr_ljungbox
from pmdarima import auto_arima

PROJECT_ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")

C_OK   = "#5cb85c"
C_WARN = "#f0ad4e"
C_FAIL = "#d9534f"
C_HEAD = "#1c2833"

print("Setup OK")

---
## 1. Estabilidad de Factores Chain Ladder

Un factor CL es confiable si:
- **CV < 0.3** (coeficiente de variación bajo entre cohortes)
- **N >= 5** (suficientes cohortes para estimarlo)

Si un factor no cumple, la proyección CL para esa transición es **poco confiable**.

In [ ]:
# General
diag = pd.read_csv(os.path.join(PROCESSED_DIR, "diagnostico_cl.csv"), sep=";", decimal=",")

def hl_diag(row):
    if row["estable"]:
        return [f"background-color:{C_OK};color:white;font-weight:bold"] * len(row)
    return [f"background-color:{C_FAIL};color:white;font-weight:bold"] * len(row)

print("FACTORES CL — GENERAL (sin niveles)")
display(
    diag.style.apply(hl_diag, axis=1)
    .format({"factor_cl": "{:.4f}", "cv_factores": "{:.3f}"})
    .set_caption("Verde = estable (CV<0.3, N>=5) | Rojo = revisar")
    .set_table_styles([{"selector": "caption", "props": "font-weight:bold;font-size:13px"}])
)

n_ok = diag["estable"].sum()
print(f"\nResumen: {n_ok}/{len(diag)} transiciones estables ({n_ok/len(diag)*100:.0f}%)")

In [ ]:
# Por niveles
diag_niv = pd.read_csv(os.path.join(PROCESSED_DIR, "diagnostico_cl_niveles.csv"), sep=";", decimal=",")

# Solo transiciones hasta MOB 18
diag_niv_18 = diag_niv[diag_niv["transicion"].apply(lambda x: int(x.split("->")[0]) < 18)].copy()

# Resumen por segmento
resumen = diag_niv_18.groupby("segmento").agg(
    total=("estable", "count"),
    estables=("estable", "sum"),
).assign(pct_estable=lambda df: df["estables"] / df["total"] * 100)

def hl_pct(row):
    pct = row["pct_estable"]
    if pct >= 80: bg = C_OK
    elif pct >= 60: bg = C_WARN
    else: bg = C_FAIL
    return [f"background-color:{bg};color:white;font-weight:bold"] * len(row)

print("FACTORES CL — POR SEGMENTO (transiciones hasta MOB 18)")
display(
    resumen.style.apply(hl_pct, axis=1)
    .format({"pct_estable": "{:.0f}%"})
    .set_caption("Verde >=80% estable | Naranja >=60% | Rojo <60%")
    .set_table_styles([{"selector": "caption", "props": "font-weight:bold;font-size:13px"}])
)

# Heatmap de CV
pivot_cv = diag_niv_18.pivot_table(index="segmento", columns="transicion", values="cv_factores")
pivot_cv = pivot_cv.reindex(columns=sorted(pivot_cv.columns, key=lambda x: int(x.split("->")[0])))

fig, ax = plt.subplots(figsize=(16, 6))
im = ax.imshow(pivot_cv.values, cmap="RdYlGn_r", aspect="auto", vmin=0, vmax=0.5)
ax.set_xticks(range(len(pivot_cv.columns)))
ax.set_xticklabels(pivot_cv.columns, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(pivot_cv.index)))
ax.set_yticklabels(pivot_cv.index, fontsize=9)
ax.set_title("CV de factores CL por segmento y transicion (verde=estable, rojo=volatil)", fontsize=11)
plt.colorbar(im, ax=ax, label="CV", shrink=0.8)
plt.tight_layout(); plt.show()

---
## 2. Test ADF de Estacionariedad (serie MOB 1)

El test Augmented Dickey-Fuller evalúa si la serie MOB 1 tiene raíz unitaria.
- **p < 0.05**: serie estacionaria -> ARIMA con d=0 es apropiado
- **p >= 0.05**: serie NO estacionaria -> se necesita diferenciación (d>=1)

In [ ]:
# ADF general
matriz = pd.read_csv(os.path.join(PROCESSED_DIR, "matriz_vintage.csv"), sep=";", decimal=",", index_col=0)
mob1 = matriz["MOB_1"].dropna().sort_index()

adf_stat, adf_pval, used_lag, nobs, crit, icbest = adfuller(mob1, maxlag=6)
print("TEST ADF — Serie MOB 1 General")
print(f"  ADF statistic: {adf_stat:.4f}")
print(f"  p-value:       {adf_pval:.4f}")
print(f"  Lags usados:   {used_lag}")
print(f"  Observaciones: {nobs}")
for k, v in crit.items():
    print(f"  Valor critico {k}: {v:.4f}")

estacionaria = adf_pval < 0.05
print(f"\n  Conclusion: {'ESTACIONARIA' if estacionaria else 'NO ESTACIONARIA -> necesita diferenciacion'}")

# Graficos
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(mob1.index, mob1.values, "o-", markersize=3)
axes[0].set_title("Serie MOB 1"); axes[0].tick_params(axis="x", rotation=45, labelsize=6)
axes[0].set_ylabel("Indice mora"); axes[0].grid(alpha=0.3)

acf_vals = acf(mob1, nlags=12, fft=True)
axes[1].bar(range(len(acf_vals)), acf_vals, color="#2171b5")
axes[1].axhline(1.96/np.sqrt(len(mob1)), ls="--", color="red", alpha=0.7)
axes[1].axhline(-1.96/np.sqrt(len(mob1)), ls="--", color="red", alpha=0.7)
axes[1].set_title("Autocorrelacion (ACF)"); axes[1].set_xlabel("Lag")
plt.suptitle(f"Diagnostico MOB 1 — ADF p={adf_pval:.4f}", fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# ADF por segmento
mat_niv = pd.read_csv(os.path.join(PROCESSED_DIR, "matrices_vintage_niveles.csv"), sep=";", decimal=",")
segmentos = sorted(mat_niv["segmento"].unique())

adf_rows = []
for seg in segmentos:
    sub = mat_niv[mat_niv["segmento"]==seg]
    if "MOB_1" not in sub.columns: continue
    m1 = sub.set_index("cohorte")["MOB_1"].dropna().sort_index()
    if len(m1) < 8: continue
    try:
        stat, pval, *_ = adfuller(m1, maxlag=4)
        adf_rows.append({"segmento": seg, "n_obs": len(m1), "adf_stat": stat,
                         "adf_pvalue": pval, "estacionaria": pval < 0.05})
    except: pass

df_adf = pd.DataFrame(adf_rows).set_index("segmento")
def hl_adf(row):
    bg = C_OK if row["estacionaria"] else C_FAIL
    return [f"background-color:{bg};color:white;font-weight:bold"] * len(row)

print("TEST ADF — Serie MOB 1 por Segmento")
display(
    df_adf.style.apply(hl_adf, axis=1)
    .format({"adf_stat": "{:.3f}", "adf_pvalue": "{:.4f}"})
    .set_caption("Verde = estacionaria | Rojo = NO estacionaria")
    .set_table_styles([{"selector": "caption", "props": "font-weight:bold;font-size:13px"}])
)

---
## 3. Seleccion Automatica de Orden ARIMA (auto_arima)

auto_arima prueba combinaciones de (p,d,q) y selecciona la que minimiza el AIC.
Compara con el ARIMA(1,1,1) original que estaba hardcodeado.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    modelo_auto = auto_arima(mob1.values, start_p=0, max_p=3, start_q=0, max_q=3,
                             d=None, max_d=2, seasonal=False, stepwise=True,
                             suppress_warnings=True, information_criterion="aic")

print("auto_arima — MOB 1 General")
print(f"  Orden seleccionado: {modelo_auto.order}")
print(f"  AIC: {modelo_auto.aic():.2f}")
print(f"  BIC: {modelo_auto.bic():.2f}")
print(f"  Std residuos: {np.std(modelo_auto.resid(), ddof=1):.5f}")
print()
print(modelo_auto.summary())

# Residuos
resid = modelo_auto.resid()
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(resid, "o-", markersize=3); axes[0].axhline(0, color="red", ls="--")
axes[0].set_title("Residuos"); axes[0].grid(alpha=0.3)
axes[1].hist(resid, bins=15, edgecolor="black", alpha=0.7)
axes[1].set_title("Distribucion residuos")
acf_r = acf(resid, nlags=10, fft=True)
axes[2].bar(range(len(acf_r)), acf_r, color="#2171b5")
axes[2].axhline(1.96/np.sqrt(len(resid)), ls="--", color="red", alpha=0.7)
axes[2].axhline(-1.96/np.sqrt(len(resid)), ls="--", color="red", alpha=0.7)
axes[2].set_title("ACF residuos")
plt.suptitle(f"Diagnostico auto_arima{modelo_auto.order}", fontsize=12)
plt.tight_layout(); plt.show()

# Ljung-Box
lb = acorr_ljungbox(resid, lags=[5, 10], return_df=True)
print("\nTest Ljung-Box (H0: residuos son ruido blanco):")
print(lb.to_string())
print("(p > 0.05 = residuos OK, no hay autocorrelacion residual)")

---
## 4. Propagacion de Error en la Cadena CL

Cuando se aplican N factores en cascada, el error se multiplica.
Esta celda cuantifica como crece la incertidumbre del MOB 1 al MOB 18.

In [ ]:
diag_cl = pd.read_csv(os.path.join(PROCESSED_DIR, "diagnostico_cl.csv"), sep=";", decimal=",")

mob1_mean = mob1.mean()
mob1_std  = mob1.std()

rows_prop = []
valor_acum = mob1_mean
cv_acum_sq = (mob1_std / mob1_mean) ** 2

rows_prop.append({"MOB": 1, "valor": valor_acum, "std_acum": mob1_std,
                  "cv_acum": mob1_std/mob1_mean,
                  "intervalo_inf": mob1_mean - 2*mob1_std,
                  "intervalo_sup": mob1_mean + 2*mob1_std})

for _, r in diag_cl.iterrows():
    trans = r["transicion"]
    factor = r["factor_cl"]
    cv_f = r["cv_factores"] if not np.isnan(r["cv_factores"]) else 0
    mob_dst = int(trans.split("->")[1])

    valor_acum *= factor
    cv_acum_sq += cv_f ** 2
    std_acum = valor_acum * np.sqrt(cv_acum_sq)

    rows_prop.append({
        "MOB": mob_dst, "valor": valor_acum, "std_acum": std_acum,
        "cv_acum": np.sqrt(cv_acum_sq),
        "intervalo_inf": valor_acum - 2 * std_acum,
        "intervalo_sup": valor_acum + 2 * std_acum,
    })

df_prop = pd.DataFrame(rows_prop)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(df_prop["MOB"], df_prop["valor"] * 100, "o-", color="#2171b5", label="Estimacion puntual")
ax.fill_between(df_prop["MOB"], df_prop["intervalo_inf"]*100, df_prop["intervalo_sup"]*100,
                alpha=0.25, color="#2171b5", label="+-2 std (propagado)")
ax.set_xlabel("MOB"); ax.set_ylabel("% Mora")
ax.set_title("Propagacion de error CL (cohorte promedio)")
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:.1f}%"))
ax.legend(); ax.grid(alpha=0.3)

ax2 = axes[1]
ax2.bar(df_prop["MOB"], df_prop["cv_acum"] * 100, color="#fd8d3c", alpha=0.85)
ax2.set_xlabel("MOB"); ax2.set_ylabel("CV acumulado (%)")
ax2.set_title("Coeficiente de Variacion acumulado por MOB")
ax2.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:.0f}%"))
ax2.grid(alpha=0.3)

plt.tight_layout(); plt.show()

print("\nPropagacion de error (+-2 std):")
for _, r in df_prop.iterrows():
    print(f"  MOB_{int(r['MOB']):2d}: {r['valor']*100:6.2f}% +- {r['std_acum']*100:5.2f}%  (CV={r['cv_acum']*100:.1f}%)")

---
## 5. Resumen y Recomendaciones

| Aspecto | Que evalua | Criterio de alerta |
|---|---|---|
| **Factores CL** | Estabilidad entre cohortes | CV > 0.3 o N < 5 |
| **Test ADF** | Estacionariedad de MOB 1 | p >= 0.05 -> no estacionaria |
| **auto_arima** | Seleccion de orden optimo | AIC/BIC comparativo |
| **Ljung-Box** | Residuos = ruido blanco | p < 0.05 -> modelo inadecuado |
| **Error propagado** | Incertidumbre acumulada | CV > 30% -> intervalo muy amplio |